In [20]:
from constants import users_list, data_path
from lib import spoti, genre_normalizer, plotting, preprocessing, dimensionality_reduction

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import json
import os
from sklearn.decomposition import PCA
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import mean_squared_error
import random
import time
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors, NeighborhoodComponentsAnalysis
from sklearn.metrics import pairwise_distances
import pickle
import numpy as np
from typing import List, Dict, Tuple, Optional, Any, Callable
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from IPython.display import display, HTML

# Import data

In [21]:
df = spoti.load_all_tracks(
    base_path=data_path.DATA_PATH,
    users=users_list.USERS,
    load_spotify_tracks=False,
    penality_factors={"short_term": 1, "medium_term": 0.5, "long_term": 0.1},
)
df

,album,artists,available_markets,disc_number,duration_ms,explicit,external_ids,external_urls,href,id,...,time_range,affinity,username,release_year,normalized_genres,added_at,episode,track,added_by,playlist_id
0,"{'album_type': 'SINGLE', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,196426,False,{'isrc': 'USSM12301260'},{'spotify': 'https://open.spotify.com/track/75...,https://api.spotify.com/v1/tracks/75rqqKvzJCGv...,75rqqKvzJCGv2oq9C4yFDt,...,medium_term,0.50,jaslkh,2023,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
1,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,137533,True,{'isrc': 'USSM12109218'},{'spotify': 'https://open.spotify.com/track/2F...,https://api.spotify.com/v1/tracks/2FYGZDfsAnNs...,2FYGZDfsAnNsrm1gVbyKnG,...,medium_term,0.49,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
2,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,162906,True,{'isrc': 'USSM12109222'},{'spotify': 'https://open.spotify.com/track/4k...,https://api.spotify.com/v1/tracks/4kroNlz8BTfs...,4kroNlz8BTfswE4M0i3YCh,...,medium_term,0.48,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
3,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,89749,False,{'isrc': 'USSM12208854'},{'spotify': 'https://open.spotify.com/track/2N...,https://api.spotify.com/v1/tracks/2N3YZ075lq9z...,2N3YZ075lq9z1ObaAiX6l1,...,medium_term,0.47,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
4,"{'album_type': 'SINGLE', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,174044,False,{'isrc': 'USSM12300114'},{'spotify': 'https://open.spotify.com/track/2S...,https://api.spotify.com/v1/tracks/2SiAcexM2p1y...,2SiAcexM2p1yX6joESbehd,...,medium_term,0.46,jaslkh,2023,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12424,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AL, AM, AT, AZ, BA, BE, BG, BY, CH, CW, C...",1,251880,False,{'isrc': 'GBN9Y1100001'},{'spotify': 'https://open.spotify.com/track/3z...,https://api.spotify.com/v1/tracks/3z7dWKRsjDNM...,3z7dWKRsjDNM24ohLKZBnA,...,NaN,NaN,dany,1967,"[rock, rock, rock, rock, rock, rock, rock]",2022-12-30 08:42:31+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12425,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,193853,False,{'isrc': 'GBLTP1700005'},{'spotify': 'https://open.spotify.com/track/1V...,https://api.spotify.com/v1/tracks/1VofMhhL98pe...,1VofMhhL98pewltVGBSmCW,...,NaN,NaN,dany,2017,"[rock, rock, rock, rock, rock]",2020-08-10 14:53:07+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12426,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,295493,False,{'isrc': 'GBLTP1700011'},{'spotify': 'https://open.spotify.com/track/0K...,https://api.spotify.com/v1/tracks/0KE7apgczHNY...,0KE7apgczHNYiXIvMUY0Fc,...,NaN,NaN,dany,2017,"[rock, rock, rock, rock, rock]",2020-08-10 14:53:13+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12427,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,467306,False,{'isrc': 'GBLTP1700014'},{'spotify': 'https://open.spotify.com/track/6v...,https://api.spotify.com/v1/tracks/6vbRA9yAAgIX...,6vbRA9yAAgIXtDlmhyNqPq,...,NaN,N

# Logistic Matrix Factorization for Implicit Feedback Data (Logistic MF)
From [Christopher C. Johnson - Logistic Matrix Factorization for Implicit Feedback Data](https://web.stanford.edu/~rezab/nips2014workshop/submits/logmat.pdf)

## Setup the dataset

In [22]:
df_matrix_mf = df.copy()
df_matrix_mf = df_matrix_mf[df["type"] == "top_track"]
df_matrix_mf["username"] = df_matrix_mf["username"].astype("category")
df_matrix_mf["id"] = df_matrix_mf["id"].astype("category")
df_matrix_mf[["username", "id", "affinity"] + spoti.NUMERICAL_FEATURES]
df_matrix_mf["affinity"] *= 100
df_matrix_mf["affinity"]

0        50.0
1        49.0
2        48.0
3        47.0
4        46.0
         ... 
12318     1.0
12319     0.8
12320     0.6
12321     0.4
12322     0.2
Name: affinity, Length: 1200, dtype: float64

In [23]:
# Create the matrix dataset
class MatrixDataset:
    _matrix: np.ndarray

    @property
    def matrix(self) -> np.ndarray:
        return self._matrix

    def __init__(self, num_users: int, num_items: int):
        # A matrix of shape (num_users, num_items)
        self._matrix = np.zeros((num_users, num_items))

    def add_interaction(self, user_id: int, item_id: int, value: float):
        """
        Adds a new interaction to the matrix.
        :param user_id: The user id.
        :param item_id: The item id.
        :param value: The value of the interaction.
        """
        self._matrix[user_id, item_id] = value

    def fill_from_df(self, users: pd.Series, items: pd.Series, values: pd.Series):
        """
        Fills the matrix from a dataframe.
        :param df: The dataframe.
        :param user_col: The user column.
        :param item_col: The item column.
        :param value_col: The value column.
        """
        assert (
            len(users) == len(items) == len(values)
        ), "The length of the users, items and values must be the same."

        for user, item, value in zip(users, items, values):
            self.add_interaction(user, item, value)

    def __str__(self):
        return str(self._matrix)

In [24]:
num_users = len(df_matrix_mf["username"].unique())
num_items = len(df_matrix_mf["id"].unique())

In [25]:
matrix_mf = MatrixDataset(num_users, num_items)
matrix_mf.fill_from_df(df_matrix_mf["username"].cat.codes, df_matrix_mf["id"].cat.codes, df_matrix_mf["affinity"])
R = matrix_mf.matrix
R

array([[ 0. ,  0. , 62. , ...,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. , ...,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. , ...,  0. ,  0. , 54. ],
       ...,
       [ 0. ,  0. ,  0. , ...,  0. ,  0. ,  0. ],
       [ 4.4, 30. ,  0. , ...,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. , ..., 46. ,  0. ,  0. ]])

In [26]:
def convert_to_ids(values: List[str], column: str) -> List[int]:
    """
    Gets the user ids from the usernames.
    :param usernames: The usernames.
    :return: The user ids.
    """
    return df_matrix_mf[df_matrix_mf[column].isin(values)][column].cat.codes.tolist()

def retrieve_value_from_ids(ids: List[int], column: str) -> str:
    """
    Gets the value from the ids.
    :param ids: The ids.
    :return: The value.
    """
    return df_matrix_mf[df_matrix_mf[column].cat.codes.isin(ids)][column].tolist()

def get_df_rows_from_ids(ids: List[int], column: str, search_in: pd.DataFrame) -> pd.DataFrame:
    """
    Gets the dataframe rows from the ids.
    :param ids: The ids.
    :return: The dataframe rows.
    """
    return search_in[search_in[column].cat.codes.isin(ids)]

In [27]:
# Create a matrix U that contains the index that sorts the users by their affinity
I = np.argsort(matrix_mf.matrix, axis=1)
I.shape

(8, 923)

In [28]:
def compute_alpha(matrix: np.ndarray) -> np.ndarray:
    """
    Computes the alpha matrix.
    :param matrix: The matrix.
    :param k: The number of neighbors.
    :return: The alpha matrix.
    """
    num_items = matrix.shape[1] * matrix.shape[0]
    sum_matrix = matrix.flatten().sum()
    number_of_zeros = num_items - np.count_nonzero(matrix)
    alpha = number_of_zeros / sum_matrix
    return alpha

alpha = compute_alpha(matrix_mf.matrix)
print(alpha)
R *= alpha
R

0.264317361339022


array([[ 0.        ,  0.        , 16.3876764 , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        , 14.27313751],
       ...,
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 1.16299639,  7.92952084,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ..., 12.15859862,
         0.        ,  0.        ]])

_Let $l_{u,i}$ denote the event that user $u$ has chosen to interact with item $i$ (user $u$ prefers item $i$). Then, we can let the probability of this event occurring be distributed according to a logistic function parameterized by the sum of the inner product of user and item latent factor vectors and user and item biases._
$$
p(l_{ui} | x_u, y_i, \beta_i, \beta_j) = \frac{\exp(x_uy_i^T + \beta_u + \beta_i)}{1 + \exp(x_uy_i^T + \beta_u + \beta_i)}
$$



The log posterior probability of $x_u, y_i, \beta_u, \beta_i$ given the observed data $\mathbf{R}$ is given by:
$$
\log p(\mathbf{X}, \mathbf{Y}, \beta | \mathbf{R}) = \sum_{u,i} \alpha r_{ui} (x_u^T y_i + \beta_u + \beta_i) - (1 + \alpha r_{ui}) \log(1 + \exp(x_u^T y_i + \beta_u + \beta_i)) - \frac{\lambda}{2} ||x_u||^2 - \frac{\lambda}{2} ||y_i||^2
$$

In [29]:
def log_posterior(
    X: torch.Tensor,
    Y: torch.Tensor,
    beta_u: torch.Tensor,
    beta_i: torch.Tensor,
    R: torch.Tensor,
    alpha: float,
    lambd: float,
) -> torch.Tensor:
    """
    Computes the log posterior of the model, which is:
    :param X: The latent vectors of the users.
    :param Y: The latent vectors of the items.
    :param beta_u: The bias of the users.
    :param beta_i: The bias of the items.
    :param R: The matrix of interactions.
    :param alpha: The alpha parameter.
    :param lambd: The lambda parameter.
    :return: The log posterior.
    """
    # Compute the first term
    term1 = alpha * R * (torch.matmul(X, Y.t()) + beta_u[:, None] + beta_i)
    term2 = (1 + alpha * R) * torch.log1p(torch.exp(torch.matmul(X, Y.t()) + beta_u[:, None] + beta_i))
    regularization = (lambd / 2) * (torch.norm(X, p=2) ** 2 + torch.norm(Y, p=2) ** 2)
    return torch.sum(term1 - term2) - regularization

In [30]:
def mpr(I: torch.Tensor, R: torch.Tensor) -> float:
    """
    Compute the Mean Percentile Ranking (MPR) using a sorted index matrix.

    :param I: Matrix of sorted indices of items for each user.
    :param R: Rating or interaction matrix.
    :return: MPR value.
    """
    num_users, num_items = R.shape
    total_interactions = torch.sum(R)

    # Initialize MPR
    mpr = 0.0

    # Iterate over each user
    for u in range(num_users):
        # Get the indices of the items in sorted order for this user
        sorted_indices = I[u]

        # Calculate the rank for each item
        for i in range(num_items):
            item_index = sorted_indices[i]
            rank = i / num_items  # Percentile rank
            mpr += R[u, item_index] * rank

    # Normalize by the total number of interactions
    mpr /= total_interactions

    return mpr

In [31]:
def compute_predictions(
    X: torch.Tensor,
    Y: torch.Tensor,
    beta_u: torch.Tensor,
    beta_i: torch.Tensor,
) -> torch.Tensor:
    """
    Computes the predictions of the model.
    :param X: The latent vectors of the users.
    :param Y: The latent vectors of the items.
    :param beta_u: The bias of the users.
    :param beta_i: The bias of the items.
    :param R: The matrix of interactions.
    :return: The predictions.
    """
    return torch.matmul(X, Y.t()) + beta_u[:, None] + beta_i

In [32]:
def train(
    num_latent: int,
    lambd: float,
    alpha: float,
    learning_rate: float,
    epochs: int,
):
    """
    Trains the model.
    :param num_latent: The number of latent factors.
    :param lambd: The lambda parameter.
    :param alpha: The alpha parameter.
    :param learning_rate: The learning rate.
    :param epochs: The number of epochs.
    """
    mprs = []  # Record MPRs for each epoch

    # Initialize user and item latent factor matrices and bias vectors
    R = torch.tensor(matrix_mf.matrix, dtype=torch.float32)
    X = torch.randn(num_users, num_latent, requires_grad=True)
    Y = torch.randn(num_items, num_latent, requires_grad=True)
    beta_u = torch.randn(num_users, requires_grad=True)
    beta_i = torch.randn(num_items, requires_grad=True)

    # The objective is to minimize the negative log posterior
    optimizer = optim.Adagrad([X, Y, beta_u, beta_i], lr=learning_rate)

    # Training loop
    for epoch in tqdm(range(epochs)):
        # Zero out gradients
        optimizer.zero_grad()

        # Compute the loss
        loss = -log_posterior(X, Y, beta_u, beta_i, R, alpha, lambd)

        # Backpropagate
        loss.backward()

        # Update the parameters
        optimizer.step()

        # Make predictions
        predictions = compute_predictions(X, Y, beta_u, beta_i)

        # Compute MPR
        I = torch.argsort(predictions, descending=True, dim=1)
        mpr_value = mpr(I, R)

        # Record MPR
        mprs.append(mpr_value)

        # Print progress
        print(f"Epoch {epoch + 1} - Loss: {loss.item():.4f} - MPR: {mpr_value:.4f}")

    return X, Y, beta_u, beta_i, mprs

In [33]:
num_latent = 10
X, Y, beta_u, beta_i, mprs = train(
    num_latent=num_latent,
    lambd=0.1,
    alpha=alpha,
    learning_rate=0.1,
    epochs=100,
)

  2%|▏         | 2/100 [00:00<00:06, 14.52it/s]

Epoch 1 - Loss: 13199.3457 - MPR: 0.4532
Epoch 2 - Loss: 10517.6914 - MPR: 0.4199


  4%|▍         | 4/100 [00:00<00:05, 16.31it/s]

Epoch 3 - Loss: 9093.4580 - MPR: 0.3899
Epoch 4 - Loss: 8121.9048 - MPR: 0.3632


  6%|▌         | 6/100 [00:00<00:05, 16.96it/s]

...

Epoch 95 - Loss: 2014.5874 - MPR: 0.0331
Epoch 96 - Loss: 2009.3009 - MPR: 0.0330


 98%|█████████▊| 98/100 [00:05<00:00, 18.22it/s]

Epoch 97 - Loss: 2004.1208 - MPR: 0.0328
Epoch 98 - Loss: 1999.0441 - MPR: 0.0327


100%|██████████| 100/100 [00:05<00:00, 17.72it/s]

Epoch 99 - Loss: 1994.0679 - MPR: 0.0326
Epoch 100 - Loss: 1989.1895 - MPR: 0.0325


In [34]:
px.line(y=mprs, title="MPR over epochs")

In [35]:
user_latent = X.detach().numpy()
item_latent = Y.detach().numpy()

if num_latent <= 3:
    # Add the latent vectors to the dataframe
    df_matrix_mf["user_latent"] = df_matrix_mf["username"].cat.codes.apply(
        lambda x: user_latent[x]
    )
    df_matrix_mf["item_latent"] = df_matrix_mf["id"].cat.codes.apply(
        lambda x: item_latent[x]
    )

    # Add the biases to the dataframe
    df_matrix_mf["user_bias"] = df_matrix_mf["username"].cat.codes.apply(
        lambda x: beta_u[x].item()
    )
    df_matrix_mf["item_bias"] = df_matrix_mf["id"].cat.codes.apply(
        lambda x: beta_i[x].item()
    )

    for i in range(num_latent):
        df_matrix_mf[f"user_latent_{i}"] = df_matrix_mf["user_latent"].apply(lambda x: x[i])
        df_matrix_mf[f"latent_{i}"] = df_matrix_mf["item_latent"].apply(lambda x: x[i])

    fig = plotting.plot_latent_space(
        df_matrix_mf,
        color=df_matrix_mf["username"],
        text=df_matrix_mf["username"],
        title="User latent space",
        latent_columns=["latent_0", "latent_1", "latent_2"],
    )
    fig.show()

    df_user_latent = df_matrix_mf.drop_duplicates(subset=["username"])
    fig = plotting.plot_latent_space(
        df_user_latent,
        color=df_user_latent["username"],
        text=df_user_latent["username"],
        title="User latent space",
        latent_columns=["user_latent_0", "user_latent_1", "user_latent_2"],
    )
    fig.show()

In [44]:
# Choose a user
user = "jaslkh"
user_id = convert_to_ids([user], "username")[0]

# Make predictions for the user
predictions = compute_predictions(X, Y, beta_u, beta_i)

# Get the top N predictions
N = 300
I = torch.argsort(predictions[user_id], descending=True)[:N]
I = I.detach().numpy()

# Get the top 10 predictions
top_predictions = get_df_rows_from_ids(I, "id", df_matrix_mf)
top_predictions[["id"] + spoti.PRETTY_PRINT_FEATURES][130:].head(50)

,id,username,artists_names,name,release_year,popularity,danceability,energy,speechiness,acousticness,instrumentalness,liveness,valence,tempo,loudness,duration_ms,release_year,popularity
134,4ffoWIfe9UWfyuJT5I0I7L,jaslkh,Johan Papaconstantino,Les mots bleus,2019,58,0.790,0.484,0.1140,0.271000,0.012900,0.1270,0.4310,171.988,-8.267,250893,2019,58
136,38VyljyWXnVxtYWSiH5Hzc,jaslkh,Ahmed Spins,Anchor Point,2022,1,0.681,0.787,0.0354,0.390000,0.532000,0.0928,0.3900,123.025,-9.800,359024,2022,1
137,0vrDGR5ZjdDIBUuDep6yXT,jaslkh,anees,leave me,2022,66,0.669,0.697,0.2280,0.093400,0.000000,0.0864,0.5350,78.709,-6.299,220539,2022,66
146,25xa84ZW8Wy3cHvRUzpvrY,jaslkh,Mandragora,Slytherin,2020,54,0.729,0.406,0.0822,0.109000,0.565000,0.2700,0.0388,163.998,-8.831,210731,2020,54
147,34LI7rwi9H8w2S5KTHnv1M,jaslkh,Manuel Turizo,Quiéreme Mientras Se Pueda,2020,68,0.793,0.782,0.0582,0.447000,0.000000,0.1240,0.8100,143.860,-3.945,191945,2020,68
1505,4yXKLDrHYrffSVmQOFdbDA,owen,Mom Jeans.,Scott Pilgrim vs. My GPA,2016,58,0.296,0.494,0.0370,0.071500,0.050700,0.1080,0.1750,163.669,-10.490,239597,2016,58
1506,141alNiSd9vG4Lb22BLHWM,owen,Mom Jeans.,What's Up?,2022,60,0.512,0.933,0.0413,0.000109,0.000199,0.2740,0.8280,84.488,-5.598,141908,2022,60
1507,19aUuDd6udp1ACNo9t3IuZ,owen,Mom Jeans.,Death Cup,2016,62,0.353,0.749,0.0411,0.092600,0.000553,0.1660,0.5740,85.983,-9.339,276000,2016,62
1508,1AMADyXgIWayh5vXLZo2qF,owen,Basement,Covet,2012,78,0.428,0.799,0.0468,0.001390,0.074200,0.0788,0.2280,139.052,-6.265,227340,2012,78
1509,73NKI74L085oOkIxyY9sJ1,owen,Basement,Covet - alt version,2022,48,0.556,0.471,0.0256,0.001420,0.000029,0.0799,0.2990,76.516,-7.444,198038,2022,48
